In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

FINAL_PKL = "C:/Users/asus/Desktop/acs_ssc_lasso_input.pkl"
OUT_PKL   = "C:/Users/asus/Desktop/acs_ssc_predicted_v2.pkl"

df = pd.read_pickle(FINAL_PKL)
print(f"Loaded: {df.shape}  |  Years: {sorted(df['year'].unique())}")

Loaded: (37907, 20)  |  Years: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]


In [4]:
# ── Fix missing sp_male ──────────────────────────────────────────
# build_data_V2.py omits sp_male from keep_cols.

if "sp_male" not in df.columns:
    df["sp_male"] = df["r_male"]
    print("Derived sp_male = r_male  (same-sex sample — sex is identical)")

# Sanity check: confirm all required feature columns are present
needed = [
    "r_agegroup", "r_edugroup", "r_race", "r_occbroad", "r_degbroad", "r_male",
    "sp_agegroup", "sp_edugroup", "sp_race", "sp_occbroad", "sp_degbroad", "sp_male",
    "c_children", "statefip", "r_incearn", "sp_incearn"
]
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Still missing columns: {missing}")
print("All required columns present.")

All required columns present.


In [5]:
# ── Feature builder ──────────────────────────────────────────────
def build_features(person_df: pd.DataFrame) -> pd.DataFrame:
    """
    Expects columns already renamed to the canonical names:
      r_agegroup, r_edugroup, r_race, r_occbroad, r_degbroad,
      r_male, statefip, c_children
    Returns dummy-expanded + pairwise-interaction matrix.
    """
    cat_cols = ["r_agegroup", "r_edugroup", "r_race",
                "r_occbroad", "r_degbroad", "statefip"]

    base = pd.get_dummies(
        person_df[cat_cols + ["r_male", "c_children"]],
        columns=cat_cols,
        drop_first=True,
        dtype=float
    )

    # Pairwise interactions — keep only non-trivial (sum >= 10)
    base_cols = list(base.columns)
    interact_parts = []
    for i, c1 in enumerate(base_cols):
        for c2 in base_cols[i + 1:]:
            s = base[c1] * base[c2]
            if s.sum() >= 10:
                interact_parts.append(s.rename(f"{c1}_x_{c2}"))

    if interact_parts:
        X = pd.concat([base, pd.concat(interact_parts, axis=1)], axis=1)
    else:
        X = base

    return X.astype(float)


print("Feature builder defined.")

Feature builder defined.


In [ ]:
def predict_income_for_role(df, train_mask, prefix="r", cv=10, max_iter=10_000):
    earn_col = f"{prefix}_incearn"
    

    X_full = build_features(df)
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_full)
    X_train = X_sc[train_mask.values]

    # ── Step 1: P(earn > 0) ──────────────────────────────────────
    y1 = (df[earn_col] > 0).astype(float).values
    y1_train = y1[train_mask.values]

    m1 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m1.fit(X_train, y1_train)

    hat_pos_prob = m1.predict(X_sc)
    
   
    target_share = y1.mean()
    threshold = np.quantile(hat_pos_prob, 1 - target_share)
    hat_pos = (hat_pos_prob >= threshold).astype(float)

    # ── Step 2: earn in levels | earn > 0 ────────────────────────
    pos_train = train_mask.values & (df[earn_col].values > 0)
    
    
    y2_train = df.loc[pos_train, earn_col].values
    X2_train = X_sc[pos_train]

    m2 = LassoCV(cv=cv, max_iter=max_iter, n_jobs=-1, random_state=42)
    m2.fit(X2_train, y2_train)

    hat_earn_lvl = m2.predict(X_sc)
    hat_earn_lvl = np.maximum(hat_earn_lvl, 0)  # 防止负值

    # Final
    hat_incearn = np.where(hat_pos == 1, hat_earn_lvl, 0.0)

    df = df.copy()
    df[f"hat_pos_{prefix}"]     = hat_pos
    df[f"hat_earn_{prefix}"]    = hat_earn_lvl
    df[f"hat_incearn_{prefix}"] = hat_incearn
    return df

In [9]:
# ── Run — Respondent ────────────────────────────────────────────
train_mask = df["year"] == 2012
print(f"Training sample (2012): N={train_mask.sum():,}")

df = predict_income_for_role(df, train_mask, prefix="r")

Training sample (2012): N=5,070


In [10]:
# ── Run — Spouse / partner ──────────────────────────────────────
df = predict_income_for_role(df, train_mask, prefix="sp")

d:\anaconda\envs\dl\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 8.525e-02, tolerance: 5.553e-02
  model = cd_fast.enet_coordinate_descent_gram(


In [11]:
# ── Couple-level aggregates ──────────────────────────────────────
df["hat_c_incearn"] = df["hat_incearn_r"] + df["hat_incearn_sp"]

# Primary earner share = max(r, sp) / total  (matches paper's Table 2)
total = df["hat_c_incearn"].replace(0, np.nan)
df["hat_c_split"] = (
    np.maximum(df["hat_incearn_r"], df["hat_incearn_sp"]) / total
).fillna(0.5)

# Also store the 5-polynomial terms — needed for IV income controls
for i in range(1, 6):
    df[f"hat_c_incearn{i}"] = (df["hat_c_incearn"] / 10_000) ** i
    df[f"hat_c_split{i}"]   = df["hat_c_split"] ** i

print("Couple-level columns added.")

Couple-level columns added.


In [14]:
# ── Diagnostics vs Table 2 ───────────────────────────────────────
print("="*60)
print("DIAGNOSTICS vs. Table 2")
print("="*60)

print("\n--- Share positive earnings ---")
print(f"  Respondent  reported={(df['r_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_r'].mean():.3f}  "
      f"(paper predicted: 0.963 mar / 0.969 coh)")
print(f"  Spouse      reported={(df['sp_incearn']>0).mean():.3f}  "
      f"predicted={df['hat_pos_sp'].mean():.3f}")

print("\n--- Couple predicted total earnings ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    print(f"  {label}: {sub['hat_c_incearn'].mean():>10,.0f}  "
          f"(paper: mar≈110,729  coh≈102,953)")

print("\n--- Couple predicted earnings split ---")
for label, mask in [("Married", df["sscouple_mar"]),
                    ("Cohabiting", df["sscouple_coh"])]:
    sub = df[mask]
    pos = sub["hat_c_incearn"] > 0
    print(f"  {label}: {sub.loc[pos,'hat_c_split'].mean():.3f}  "
          f"(paper: mar≈0.648  coh≈0.641)")

print("\n--- Correlation reported vs predicted (positive earners) ---")
for pfx, label in [("r", "Respondent"), ("sp", "Spouse")]:
    both = (df[f"{pfx}_incearn"] > 0) & (df[f"hat_incearn_{pfx}"] > 0)
    r = np.corrcoef(df.loc[both, f"{pfx}_incearn"],
                    df.loc[both, f"hat_incearn_{pfx}"])[0, 1]
    print(f"  {label}: r={r:.3f}")

DIAGNOSTICS vs. Table 2

--- Share positive earnings ---
  Respondent  reported=0.905  predicted=0.905  (paper predicted: 0.963 mar / 0.969 coh)
  Spouse      reported=0.863  predicted=0.863

--- Couple predicted total earnings ---
  Married:    108,523  (paper: mar≈110,729  coh≈102,953)
  Cohabiting:    104,956  (paper: mar≈110,729  coh≈102,953)

--- Couple predicted earnings split ---
  Married: 0.654  (paper: mar≈0.648  coh≈0.641)
  Cohabiting: 0.626  (paper: mar≈0.648  coh≈0.641)

--- Correlation reported vs predicted (positive earners) ---
  Respondent: r=0.478
  Spouse: r=0.262
